# Feature engineering

Primero haremos modelos para cada familia de productos y ver que tal funciona

In [140]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [141]:
TRAIN_CSV_PATH = '../data/raw/store-sales-time-series-forecasting/train.csv'

In [142]:
train = pd.read_csv(TRAIN_CSV_PATH, parse_dates=['date'])
train['date'] = pd.to_datetime(train['date'])

In [143]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [144]:
grouped_train = train.groupby(['family', 'date', 'store_nbr']).agg({'sales': 'sum', 'onpromotion': 'sum'}).reset_index()

grouped_train.head()

,family,date,store_nbr,sales,onpromotion
0,AUTOMOTIVE,2013-01-01,1,0.0,0
1,AUTOMOTIVE,2013-01-01,2,0.0,0
2,AUTOMOTIVE,2013-01-01,3,0.0,0
3,AUTOMOTIVE,2013-01-01,4,0.0,0
4,AUTOMOTIVE,2013-01-01,5,0.0,0


In [145]:
families = grouped_train['family'].value_counts().index

In [149]:
# Aplicamos OHE a store_nbr
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' para evitar la trampa de variables ficticias

ohe.fit(grouped_train[['store_nbr']])

,categories,'auto'
,drop,'first'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


array(['store_nbr_2', 'store_nbr_3', 'store_nbr_4', 'store_nbr_5',
       'store_nbr_6', 'store_nbr_7', 'store_nbr_8', 'store_nbr_9',
       'store_nbr_10', 'store_nbr_11', 'store_nbr_12', 'store_nbr_13',
       'store_nbr_14', 'store_nbr_15', 'store_nbr_16', 'store_nbr_17',
       'store_nbr_18', 'store_nbr_19', 'store_nbr_20', 'store_nbr_21',
       'store_nbr_22', 'store_nbr_23', 'store_nbr_24', 'store_nbr_25',
       'store_nbr_26', 'store_nbr_27', 'store_nbr_28', 'store_nbr_29',
       'store_nbr_30', 'store_nbr_31', 'store_nbr_32', 'store_nbr_33',
       'store_nbr_34', 'store_nbr_35', 'store_nbr_36', 'store_nbr_37',
       'store_nbr_38', 'store_nbr_39', 'store_nbr_40', 'store_nbr_41',
       'store_nbr_42', 'store_nbr_43', 'store_nbr_44', 'store_nbr_45',
       'store_nbr_46', 'store_nbr_47', 'store_nbr_48', 'store_nbr_49',
       'store_nbr_50', 'store_nbr_51', 'store_nbr_52', 'store_nbr_53',
       'store_nbr_54'], dtype=object)

In [151]:
dataframes = {}


for family in families:
    family_df = grouped_train[grouped_train['family'] == family].copy()
    family_df.set_index('date', inplace=True)
    ohe_store = ohe.transform(family_df[['store_nbr']])
    family_df = pd.concat([family_df, pd.DataFrame(ohe_store, index=family_df.index, columns=ohe.get_feature_names_out(['store_nbr']))], axis=1)
    family_df.drop(columns=['store_nbr'], inplace=True)
    dataframes[family] = family_df
    

dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion,store_nbr_2,store_nbr_3,store_nbr_4,store_nbr_5,store_nbr_6,store_nbr_7,store_nbr_8,...,store_nbr_45,store_nbr_46,store_nbr_47,store_nbr_48,store_nbr_49,store_nbr_50,store_nbr_51,store_nbr_52,store_nbr_53,store_nbr_54
date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,AUTOMOTIVE,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01,AUTOMOTIVE,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01,AUTOMOTIVE,0.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01,AUTOMOTIVE,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01,AUTOMOTIVE,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [152]:
#Ver si hay valores nulos en los dataframes
for family, df in dataframes.items():
    if df.isnull().values.any():
        print(f"Missing values found in {family} dataframe.")
    else:
        print(f"No missing values in {family} dataframe.")

No missing values in AUTOMOTIVE dataframe.
No missing values in BABY CARE dataframe.
No missing values in BEAUTY dataframe.
No missing values in BEVERAGES dataframe.
No missing values in BOOKS dataframe.
No missing values in BREAD/BAKERY dataframe.
No missing values in CELEBRATION dataframe.
No missing values in CLEANING dataframe.
No missing values in DAIRY dataframe.
No missing values in DELI dataframe.
No missing values in EGGS dataframe.
No missing values in FROZEN FOODS dataframe.
No missing values in GROCERY I dataframe.
No missing values in GROCERY II dataframe.
No missing values in HARDWARE dataframe.
No missing values in HOME AND KITCHEN I dataframe.
No missing values in HOME AND KITCHEN II dataframe.
No missing values in HOME APPLIANCES dataframe.
No missing values in HOME CARE dataframe.
No missing values in LADIESWEAR dataframe.
No missing values in LAWN AND GARDEN dataframe.
No missing values in LINGERIE dataframe.
No missing values in LIQUOR,WINE,BEER dataframe.
No missin

In [153]:
# Restamos la fecha a la fecha mínima para obtener el número de días desde el inicio

for family, df in dataframes.items():
    df['days_since_start'] = (df.index - df.index.min()).days + 1


dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion,store_nbr_2,store_nbr_3,store_nbr_4,store_nbr_5,store_nbr_6,store_nbr_7,store_nbr_8,...,store_nbr_46,store_nbr_47,store_nbr_48,store_nbr_49,store_nbr_50,store_nbr_51,store_nbr_52,store_nbr_53,store_nbr_54,days_since_start
date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,AUTOMOTIVE,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2013-01-01,AUTOMOTIVE,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2013-01-01,AUTOMOTIVE,0.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2013-01-01,AUTOMOTIVE,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2013-01-01,AUTOMOTIVE,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [154]:
# Quitamos la columna family y onpromotion de los features ya que no es necesaria
#Guardamos primero la columna onpromotion en un diccionario para poder usarla en el modelo de XGBoost
onpromotion = {}


for family, df in dataframes.items():
    onpromotion[family] = df['onpromotion'].copy()
    df.drop(columns=['family', 'onpromotion'], inplace=True)

In [155]:
# Los partimos en train y test, dejando los últimos 90 días para test
for family, df in dataframes.items():
    train_df = df.iloc[:-90]
    test_df = df.iloc[-90:]
    dataframes[family] = (train_df, test_df)

Ahora empezamos con los datos para XGBoost  
No restaremos a los 'sales' la tendencia, lo haremos en el siguiente notebook

In [156]:
# Pondremos features de dia y mes que es
for family, (train_df, test_df) in dataframes.items():


    train_df['Month'] = train_df.index.month
    train_df['Day'] = train_df.index.day

    test_df['Month'] = test_df.index.month
    test_df['Day'] = test_df.index.day

    dataframes[family] = (train_df, test_df)

C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2002332174.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['Month'] = train_df.index.month
C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2002332174.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['Day'] = train_df.index.day
C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2002332174.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

In [157]:
#Teniendo todo alineamos la columna onpromotion con los dataframes de train y test para poder usarla en el modelo de XGBoost
for family, (train_df, test_df) in dataframes.items():

    train_onpromotion = onpromotion[family].iloc[:-90]
    test_onpromotion = onpromotion[family].iloc[-90:]

    train_df['onpromotion'] = train_onpromotion.values
    test_df['onpromotion'] = test_onpromotion.values

    dataframes[family] = (train_df, test_df)

C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2757716268.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['onpromotion'] = train_onpromotion.values
C:\Users\felix\AppData\Local\Temp\ipykernel_13436\2757716268.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['onpromotion'] = test_onpromotion.values


In [158]:
dataframes['AUTOMOTIVE'][0].head()

,sales,store_nbr_2,store_nbr_3,store_nbr_4,store_nbr_5,store_nbr_6,store_nbr_7,store_nbr_8,store_nbr_9,store_nbr_10,...,store_nbr_49,store_nbr_50,store_nbr_51,store_nbr_52,store_nbr_53,store_nbr_54,days_since_start,Month,Day,onpromotion
date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,0
2013-01-01,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,0
2013-01-01,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,0
2013-01-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,0
2013-01-01,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,0


In [159]:
# Lo guardamos los dos conjuntos de dataframes en distintos archivos csv para poder usarlos en el siguiente notebook
os.makedirs('../data/processed/LinearandXGBoost/train', exist_ok=True)
os.makedirs('../data/processed/LinearandXGBoost/test', exist_ok=True)

for family, (train_df, test_df) in dataframes.items():
    family = family.replace("/", "_")  # Reemplaza las barras por guiones bajos en el nombre de la familia
    train_df.to_csv(f'../data/processed/LinearandXGBoost/train/{family}_train.csv', index=True)
    test_df.to_csv(f'../data/processed/LinearandXGBoost/test/{family}_test.csv', index=True)